# Donut fine-tune for InBody extraction — Colab runner

Runs `cera.training.train` on Colab GPU instead of the local 8GB laptop GPU (which is compute-bound on `donut-base`'s 2560x1920 canvas — ~150s/step locally).

**Before running**: Runtime -> Change runtime type -> pick a GPU (T4 is free tier; A100 needs Colab Pro and is much faster).

The dataset cell auto-resumes: first run regenerates the 5,000-sheet synthetic dataset (~60-90 min, installs headless Chrome) and saves a zip to your Drive. Any later run (e.g. after a disconnect) finds that zip and skips straight to unzipping it — seconds instead of an hour.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "feat/ocr-inbody-extraction"

# Private repo: add a fine-grained GitHub token (Contents: Read-only, scoped
# to QeekOw/CERA) as a Colab secret named GH_TOKEN (key icon in the left
# sidebar), then run this cell.
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GH_TOKEN")
    GITHUB_REPO = f"https://{GH_TOKEN}@github.com/QeekOw/CERA.git"
except Exception:
    GITHUB_REPO = "https://github.com/QeekOw/CERA.git"  # public fallback

%cd /content
!rm -rf /content/repo
!git clone --branch $BRANCH --single-branch $GITHUB_REPO repo
%cd /content/repo
!pip install -q -e ".[training]"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_ZIP = "/content/drive/MyDrive/cera/synthetic.zip"
DATA_DIR = "/content/data/synthetic"

import os
if os.path.exists(DATA_ZIP):
    # Fast path: a prior run already generated + saved this dataset to Drive.
    print("Found saved dataset zip on Drive, skipping regeneration.")
    !mkdir -p "$DATA_DIR"
    !unzip -q "$DATA_ZIP" -d "$DATA_DIR"
else:
    # First run (or the zip step below never completed): regenerate from
    # scratch, ~60-90 min. apt's chromium-browser on Ubuntu 22.04 (Colab's
    # OS) is a snap stub that doesn't work in a container (no snapd) —
    # install real Google Chrome instead, which _find_browser() looks for.
    !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    !apt-get -qq install -y ./google-chrome-stable_current_amd64.deb
    !python -c "from pathlib import Path; from cera.training.dataset import generate_dataset; generate_dataset(Path('$DATA_DIR'))"

    # Save to Drive so a future disconnect skips regeneration entirely.
    !mkdir -p /content/drive/MyDrive/cera
    !cd "$DATA_DIR" && zip -q -r "$DATA_ZIP" .
    print("Saved dataset zip to Drive for future resumes.")

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/cera/checkpoints/donut-inbody"  # persists past session timeout
!mkdir -p "$CHECKPOINT_DIR"

In [ ]:
# T4 (16GB): batch 2, no checkpointing needed. A100 (40GB): bump batch to 4-8.
!python -m cera.training.train \
  --data-dir "$DATA_DIR" --output-dir "$CHECKPOINT_DIR" \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 3 --batch-size 2 --gradient-accumulation-steps 2 --learning-rate 3e-5

If disconnected mid-run: re-run this notebook from the top, then re-run the training cell with `--resume` appended — checkpoints save every epoch to Drive, so nothing before the last completed epoch is lost.